In [1]:
import xarray as xr
import pandas as pd
import glob
import os

In [2]:
# Folder path file MODIS 8D
folder_path = '/kaggle/input/datasets/andreh19/modis-data-8d/requested_files/'

# Ambil semua file .nc
file_list = glob.glob(os.path.join(folder_path, '*.nc'))

# Fungsi untuk ekstrak tanggal awal dari nama file
def extract_start_date(filename):
    basename = os.path.basename(filename)
    date_str = basename.split('.')[1].split('_')[0]  # '20140101'
    return pd.to_datetime(date_str, format='%Y%m%d')

# Urutkan file berdasarkan tanggal awal
file_list = sorted(file_list, key=extract_start_date)

# List untuk menampung dataset
ds_list = []

for f in file_list:
    ds = xr.open_dataset(f)
    
    # Buat dimensi time dari tanggal awal file
    start_date = extract_start_date(f)
    ds = ds.expand_dims(time=[start_date])
    
    # Jika nama variable berbeda, samakan (contoh: 'chlor_a')
    if 'x_chlor_a' in ds.variables:
        ds = ds.rename({'x_chlor_a': 'chlor_a'})
    
    ds_list.append(ds)

# Gabungkan sepanjang dimensi time
combined_ds = xr.concat(ds_list, dim='time')

# Sort by time untuk memastikan urutan benar
combined_ds = combined_ds.sortby('time')

# Simpan hasil gabungan ke file .nc
output_path = '/kaggle/working/combined_chla_2014_2025.nc'
combined_ds.to_netcdf(output_path)

print("Dataset berhasil digabung. File tersimpan di:", output_path)
print(combined_ds)

Dataset berhasil digabung. File tersimpan di: /kaggle/working/combined_chla_2014_2025.nc
<xarray.Dataset> Size: 356MB
Dimensions:  (time: 505, lat: 408, lon: 432, rgb: 3, eightbitcolor: 256)
Coordinates:
  * time     (time) datetime64[ns] 4kB 2014-01-01 2014-01-09 ... 2025-01-01
  * lat      (lat) float32 2kB 6.979 6.938 6.896 6.854 ... -9.896 -9.938 -9.979
  * lon      (lon) float32 2kB 90.02 90.06 90.1 90.15 ... 107.9 107.9 108.0
Dimensions without coordinates: rgb, eightbitcolor
Data variables:
    chlor_a  (time, lat, lon) float32 356MB 0.1617 0.1546 ... 0.0561 0.05662
    palette  (time, rgb, eightbitcolor) uint8 388kB 147 0 108 144 0 ... 0 0 0 0
Attributes: (12/61)
    product_name:                     AQUA_MODIS.20140101_20140108.L3m.8D.CHL...
    instrument:                       MODIS
    title:                            MODISA Level-3 Equidistant Cylindrical ...
    project:                          Ocean Biology Processing Group (NASA/GS...
    platform:                    

In [3]:
# JUMLAH FILE (HARI)

folder_path = '/kaggle/input/datasets/andreh19/modis-data-8d/requested_files/'
file_list = glob.glob(os.path.join(folder_path, '*.nc'))
jumlah_file = len(file_list)
print("Jumlah file .nc di folder:", jumlah_file)

Jumlah file .nc di folder: 505


In [4]:
ds = xr.open_dataset(
    "/kaggle/working/combined_chla_2014_2025.nc",chunks={"time": 30})

/tmp/ipykernel_16/3421491547.py:1: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 30. This could degrade performance. Instead, consider rechunking after loading.
  ds = xr.open_dataset(


In [5]:
ds

<xarray.Dataset> Size: 356MB
Dimensions:  (time: 505, lat: 408, lon: 432, rgb: 3, eightbitcolor: 256)
Coordinates:
  * time     (time) datetime64[ns] 4kB 2014-01-01 2014-01-09 ... 2025-01-01
  * lat      (lat) float32 2kB 6.979 6.938 6.896 6.854 ... -9.896 -9.938 -9.979
  * lon      (lon) float32 2kB 90.02 90.06 90.1 90.15 ... 107.9 107.9 108.0
Dimensions without coordinates: rgb, eightbitcolor
Data variables:
    chlor_a  (time, lat, lon) float32 356MB dask.array<chunksize=(30, 26, 27), meta=np.ndarray>
    palette  (time, rgb, eightbitcolor) uint8 388kB dask.array<chunksize=(30, 3, 256), meta=np.ndarray>
Attributes: (12/61)
    product_name:                     AQUA_MODIS.20140101_20140108.L3m.8D.CHL...
    instrument:                       MODIS
    title:                            MODISA Level-3 Equidistant Cylindrical ...
    project:                          Ocean Biology Processing Group (NASA/GS...
    platform:                         Aqua
    source:                           satellite observations from MODIS-Aqua
    ...                               ...
    processing_level:                 L3 Mapped
    cdm_data_type:                    grid
    proj4_string:                     +proj=eqc +lat_ts=0 +lat_0=0 +x_0=0 +y_...
    data_bins:                        75746
    data_minimum:                     0.023176972
    data_maximum:                     74.39555